# New vs Old GIN Code

## Packages

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import pickle
import pyarrow.parquet as pq
import random
import seaborn as sns
import torch
import torch.nn.functional as F
import umap.umap_ as umap
import warnings
from adjustText import adjust_text
from itertools import combinations
from scipy.stats import spearmanr, zscore
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from torch.nn import Identity, Linear, Module, ReLU, Sequential
from torch_geometric.data import Batch, Data, DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.utils import from_networkx
from tqdm import tqdm

## Functions

In [ ]:
# Pickle files

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()
        
# GIN

def preprocess_graphs(list_pyg):
    '''
    Applies only common attributes to PyG graph objects (some graphs have extra 'dose' attribute that required removal)
    '''
    keys_to_keep = ['x', 'edge_index', 'num_nodes', 'name', 'timepoint']
    for g in list_pyg:
        # Remove unnecessary attributes
        for key in list(g.keys()):  # <-- call keys() as a method
            if key not in keys_to_keep:
                del g[key]

        # Make x 2D
        if g.x.dim() == 1:
            g.x = g.x.view(-1, 1)
    return list_pyg

class GINEncoder(torch.nn.Module):
    '''
    GIN encoder with message passing, a convolutional layer and a global pooling layer to generate a graph embedding
    '''
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            mlp = Sequential(Linear(in_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
            conv = GINConv(mlp)
            self.convs.append(conv)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
        graph_emb = global_add_pool(x, batch)  # graph-level embedding
        return graph_emb

def info_nce_loss(z1, z2, temperature=0.5):
    '''
    Information Noise-Contrastive Estimation as a loss function
    '''
    # Normalize embeddings
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    
    batch_size = z1.size(0)
    
    # Cosine similarity matrix
    representations = torch.cat([z1, z2], dim=0)  # [2*B, D]
    sim_matrix = torch.matmul(representations, representations.T)  # [2B,2B]
    
    # Scale by temperature
    sim_matrix = sim_matrix / temperature
    
    # Labels: each sample i in z1 matches i+B in z2, and vice versa
    labels = torch.arange(batch_size, device=z1.device)
    
    # Mask out self-similarities
    mask = torch.eye(2*batch_size, device=z1.device).bool()
    sim_matrix.masked_fill_(mask, -9e15)  # very large negative number instead of -inf
    
    # Positive logits
    positives = torch.cat([torch.arange(batch_size, batch_size*2), torch.arange(0, batch_size)]).to(z1.device)
    
    # Cross-entropy
    loss = F.cross_entropy(sim_matrix, positives)
    return loss

def augment_graph_batch(batch_graphs, noise_scale=0.1):
    '''
    Graph batch augmentation
    '''
    x_aug = batch_graphs.x + noise_scale * torch.randn_like(batch_graphs.x)
    batch_aug = Batch(batch_graphs.to_data_list())
    batch_aug.x = x_aug.to(batch_graphs.x.device)
    batch_aug.edge_index = batch_graphs.edge_index
    batch_aug.batch = batch_graphs.batch
    return batch_aug

def train_contrastive_gin(list_pyg, embed_dim=64, epochs=30, batch_size=4, lr=1e-3, device=None):
    '''
    Implementaion of GIN w/ contrastive learning loss function
    '''
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

    # list_pyg = strip_graph_attributes(list_pyg)
    loader = DataLoader(list_pyg, batch_size=batch_size, shuffle=True)
    encoder = GINEncoder(input_dim=1, hidden_dim=embed_dim).to(device)
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)

    #for epoch in tqdm(range(epochs), desc = f'Training contrastive GIN', total = epochs):
    for epoch in range(epochs):
        encoder.train()
        total_loss = 0
        #for batch_graphs in tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}'):
        for batch_graphs in loader:
            batch_graphs = batch_graphs.to(device)
            batch_aug = augment_graph_batch(batch_graphs)

            z1 = encoder(batch_graphs.x, batch_graphs.edge_index, batch_graphs.batch)
            z2 = encoder(batch_aug.x, batch_aug.edge_index, batch_aug.batch)

            loss = info_nce_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        # Report loss per epoch
        #print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

    return encoder

def get_graph_embeddings(encoder, list_pyg, device=None):
    '''
    Generates graph embeddings using trained contrastive GIN
    '''
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    encoder.eval()
    embeddings = []
    graph_info = []
    with torch.no_grad():
        #for g in tqdm(list_pyg, desc="Generating embeddings"):
        for g in list_pyg:
            g = g.to(device)
            batch = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
            z = encoder(g.x, g.edge_index, batch)
            embeddings.append(z.cpu())
            graph_info.append({'name': g.name, 'timepoint': g.timepoint})
    embeddings = torch.cat(embeddings, dim=0).numpy()
    return embeddings, graph_info

## PyG Graph List

In this section:

1. `graph_cds` is a pre-generated networkX graph object made up of 901 nodes and their interations (edges). This set of nodes and edges remains the same throughout, and forms the structure onto which node signals are overlayed.

2. `df_cd` is a DataFrame containing the node signal values for the same 901 genes (rows) for 3 treatments (columns) at the 6H timepoint.

3. `pf` is a Parquet file containing ~119k 901-gene vectors (a.k.a signatures), each representing an individual drug, extracted from the CDS2 online database
    - A random number of these 119k vectors are extracted (later referred to as `NUM_ENTRIES`) and each vector is overlayed onto a copy of `graph_cds`

In [ ]:
# Load graph
graph_cds = pickle_load(INPUT + 'graph_cds.pkl')
# Load quantile CD vector data
df_cd = pd.read_csv(CDV + 'cd_quantile_landmark.csv')
# Set index
df_cd.set_index('gene_symbol', inplace = True)

# Load parquet file
pf = pq.ParquetFile(INPUT + 'df_attr.parquet')
# Load signature list
list_signatures = pf.schema.names
# Report
print(f'{len(list_signatures):,} signatures in CDS data')

In this section:

1. We define a set of filters to extract the drug signatures for one or more cell lines and timepoints

2. We then define a number of random signatures to extract from the results of the filter, and extract the name of those signatures

In [ ]:
# Define filters
CELL_LINES = ['HT29']
TIMEPOINTS = ['6H']
# Define number of random signatures
NUM_ENTRIES = 5

# Filter signatures
list_filter = [entry for entry in list_signatures if
               any(cell_line in entry for cell_line in CELL_LINES) and
               any(timepoint in entry for timepoint in TIMEPOINTS)]
# Report
print(f'{len(list_filter):,} signatures found for {CELL_LINES} cell lines and {TIMEPOINTS} timepoints')

# Sample signatures
list_random = random.sample(list_filter, NUM_ENTRIES)

In this section:

1. We extract the 6H experimental data from `df_cd` to match our filtered CDS2 data

2. We iterate through:
    - Experimental data, extracting the node signal values and assigning to a copy of `graph_cds`
    - The list of randomly selected CDS2 signatures, assigning each to a copy of `graph_cds`

This generates `list_pyg` which is a set of PyG graph objects (required format to pass to GIN), where each object represents the same graph structure with different node signals overlayed.

In [ ]:
# Define timepoint columns
list_columns = [column for column in df_cd.columns if '6h' in column]

# Initialise graph list
list_pyg = []

# Iterate through df_cd timepoint columns
for column in tqdm(list_columns, desc = 'Converting experimental data to PyG object(s)', total = len(list_columns)):

    # Get treatment name
    treatment_name = column.split('_')[2]
    # Get timepoint
    timepoint = column.split('_')[0].upper()

    # Extract column data as dictionary
    dict_column = df_cd[column].to_dict()
    
    # Copy graph_cds
    graph_column = graph_cds.copy()
    # Set node attributes
    nx.set_node_attributes(graph_column, dict_column, name = 'x')
    
    # Convert to PyG object
    pyg = from_networkx(graph_column)
    # Format 'x' attribute
    pyg.x = pyg.x.float()
    # Add treatment name
    pyg.name = f'{treatment_name}'
    # Add perturbagen timepoint
    pyg.timepoint = f'{timepoint}'
    # Append to graph list
    list_pyg.append(pyg)

# Iterate through list_random
for entry in tqdm(list_random, desc = 'Converting CDS data to PyG object(s)', total = len(list_random)):

    # Get signature ID
    signature_id = entry.split(':')[0]
    # Get cell line
    cell_line = signature_id.split('_')[1]
    # Get timepoint
    timepoint = signature_id.split('_')[2]
    # Get dose
    dose = entry.split(':')[2]

    # Get perturbagen name
    perturbagen_name = entry.split(':')[3]
    if '-666' in perturbagen_name:
        perturbagen_name = entry.split(':')[1]

    # Load signature from parquet file as dictionary
    dict_signature = pd.read_parquet(INPUT + 'df_attr.parquet', columns = [entry]).to_dict()
    dict_signature = dict_signature[entry]

    # Copy graph_cds
    graph_signature = graph_cds.copy()
    # Set node attributes
    nx.set_node_attributes(graph_signature, dict_signature, name = 'x')

    # Convert to PyG object
    pyg = from_networkx(graph_signature)
    # Format 'x' attribute
    pyg.x = pyg.x.float()
    # Add perturbagen anme
    pyg.name = f'{perturbagen_name}'
    # Add timepoint
    pyg.timepoint = f'{timepoint}'
    # Add dose
    pyg.dose = f'{dose}'
    # Add cell line
    pyg.cell = f'{cell_line}'
    # Append to graph list
    list_pyg.append(pyg)

## New GIN

In this section:

1. The GIN generated using `train_contrastive_gin()` is run multiple times (defined by `num_runs`)

2. This generates multiple sets of graph embeddings, which are centered and normalised

3. Cosine similarity between all graphs within one set of graph embeddings is calculated
    - This generates a set of cosine similarity matrices, one matrix per set of graph embeddings i.e. one similarity matrix per GIN run

These multiple results are then compared with a number of similarity metrics to extract the consensus embedding from all GIN runs; however, that code is not included here as the main objective is to assure that the GIN is producing the data/comparisons that we're interested in

In [ ]:
warnings.filterwarnings('ignore')

# Define multi-run parameters
num_runs = 25
sim_matrices = []
graph_info_list = []

# Preprocess graph list
list_pyg = preprocess_graphs(list_pyg)

for seed in tqdm(range(num_runs), desc = 'Multiple GIN runs', total = num_runs):
    torch.manual_seed(seed)
    np.random.seed(seed)

    encoder = train_contrastive_gin(list_pyg, embed_dim=64, epochs=150, batch_size=32)
    embeddings, graph_info = get_graph_embeddings(encoder, list_pyg)

    # Center and normalize
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    emb_norm = emb_centered / np.linalg.norm(emb_centered, axis=1, keepdims=True)

    sim_matrices.append(cosine_similarity(emb_norm))
    graph_info_list = graph_info  # same across runs

## Old GIN

In this section:

1. A GIN is defined where only message passing between nodes is done, with global pooling to generate graph embeddings and with no 'learning' taking place

2. In this case, the `generate_embeddings()` function allowed for manual setting of the seed for testing of GIN output given different starting seeds

This provides the example to compare to the new GIN code.

In [ ]:
# Parameters
EMBED_DIM = 64

# Define GIN encoder using identity GINConvs and fixed projection to d dimensions
class GINEncoder(Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        
        # Project scalar node feature to d-dim before aggregation
        self.initial_proj = Linear(1, embed_dim)  # assuming input is 1-d per node

        # Identity "MLPs" in GINConv — only aggregate neighbors
        mlp1 = Sequential(Identity())
        mlp2 = Sequential(Identity())

        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)

        self.lin = Identity()  # optionally project again later

    def forward(self, x, edge_index):
        if x.dim() == 1:
            x = x.view(-1, 1)  # reshape from [901] → [901, 1]
        x = self.initial_proj(x)  # shape: [n, d]
        x = self.conv1(x, edge_index)
        x = F.relu(x)  # optional
        x = self.conv2(x, edge_index)
        return self.lin(x)  # shape: [n, d] — node-level embeddings

print('Identity-mapping GIN encoder complete')

# Function to generate embeddings given a random seed
def generate_embeddings(seed):
    torch.manual_seed(seed)
    encoder = GINEncoder()
    encoder.eval()
    graph_embeddings = []

    with torch.no_grad():
        for graph in list_pyg:
            x = encoder(graph.x, graph.edge_index)
            batch = torch.zeros(x.size(0), dtype=torch.long)
            graph_emb = global_add_pool(x, batch)
            graph_embeddings.append(graph_emb)
    
    matrix = torch.cat(graph_embeddings, dim=0).numpy()
    return matrix

print('generate_embeddings function defined')